# 03b - Strategy Fine-Tuning

Supervised transfer learning from the completed broad checkpoint in 02. Use a GPU runtime for real training; CPU works for small experiments. Self-play is optional and separate.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Project and Dependencies

Keep Colab's existing CUDA PyTorch. Restart only if pip explicitly requires it.

In [ ]:
from pathlib import Path
import sys
import subprocess

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/Deep Reinforcement Learning/Chess")
if not (PROJECT_ROOT / "chess_rl").is_dir():
    raise FileNotFoundError(f"Project files are missing from {PROJECT_ROOT}. See README.md.")
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-r", str(PROJECT_ROOT / "requirements_colab.txt")])

## Run Configuration

Edit configs/default.yaml once for the workflow. Use a new run_id for a changed experiment.

In [ ]:
from chess_rl.config import load_config, prepare_directories
from chess_rl.reproducibility import metadata, read_json, atomic_json, sha256

prepare_directories(PROJECT_ROOT)
cfg = load_config(PROJECT_ROOT, "strategy.yaml")
print("Run:", cfg["run_id"])
print("Runtime:", metadata())

## Fine-Tuning Configuration

Edit strategy_finetuning in configs/strategy.yaml. Select themes and manifests; use a distinct candidate id for every specialist. The combined model and each selected specialist start independently from 02. Defaults: two head-only epochs at 1e-4 then up to eight full epochs at 3e-5; AdamW, cosine, weight decay 1e-4, patience 3 in the full phase, 512 CUDA/64 CPU. Frozen BatchNorm stays fixed. Each batch is 75% strategy and 25% broad training replay, with theme-balanced strategy sampling.

In [ ]:
import torch
from chess_rl.strategy_finetuning import load_finetuning_config, finetune_strategy
cfg = load_finetuning_config(PROJECT_ROOT)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Available training device:", device)
print(cfg["strategy_finetuning"])

## Load Base and Train or Resume

initial_selection.json from 02 supplies the base checkpoint and checksum. The architecture is loaded from that checkpoint. No random replacement is made when the base is missing. Training uses legal policy targets and side-to-move values; unlabelled rows are excluded.

In [ ]:
checkpoints = finetune_strategy(PROJECT_ROOT, cfg)
print("Completed candidates:", checkpoints)

## Saved Metrics

Candidates and resume pointers are under models/strategy/<run_id>/<candidate_id>/. Epoch checkpoints retain metrics, optimizer, scheduler, RNG and sampling state. Test data is never used for early stopping.

In [ ]:
directory = PROJECT_ROOT / "models/strategy" / cfg["run_id"]
print(read_json(directory / "completed.json"))
print("Next: optional 04a/04b, or 05a.")